# Đề tài 10 · Nhận dạng mẫu: Phân loại Âm thanh & Phát hiện Giọng nói Nhân bản Zero-Shot
## Thực nghiệm 5.0: Chẩn đoán Ngữ âm & Phân tích Động học Suy luận Mô hình Zero-Shot (Ablation Study)

**Nhóm nghiên cứu: Nhóm 05 · Chuyên ngành Khoa học Máy tính / Trí tuệ Nhân tạo**

### 1. Mục tiêu & Đặt Vấn đề Khoa học
Từ các phát hiện sai số định tính ở Thực nghiệm 4.0, thực nghiệm này thực hiện nghiên cứu loại trừ (Ablation Study) nhằm xác định nguyên nhân gốc rễ dẫn đến hiện tượng đọc sai thanh điệu hoặc thiếu âm tiết trong mô hình sinh âm thanh zero-shot OmniVoice trên tiếng Việt.

### 2. Thiết kế Thí nghiệm Loại trừ (Ablation Design)
Khảo sát trên các câu thoại điển hình xuất hiện sai số ngữ âm, thực hiện 24 lượt sinh có kiểm soát theo ma trận $3 \text{ điều kiện xử lý} \times 2 \text{ hạt ngẫu nhiên (Random Seeds)}$:
1. **Điều kiện Đối chứng (Baseline - `goc`):** Giữ nguyên định dạng chữ hoa/thường nguyên bản và kích hoạt bộ lọc hậu xử lý âm học (`postprocess_output=True`).
2. **Điều kiện Chuẩn hóa Chữ thường (`chu_thuong`):** Chuyển toàn bộ văn bản đầu vào về chữ thường không hoa để loại trừ xung đột mã hóa ngữ nghĩa (Case-sensitivity bias).
3. **Điều kiện Bỏ Hậu xử lý Âm học (`khong_xu_ly`):** Giữ nguyên văn bản nhưng vô hiệu hóa bộ lọc hậu xử lý (`postprocess_output=False`) để khảo sát tác động của bộ vocoder nội tại.

### 3. Ý nghĩa Khoa học & Sản phẩm
- Làm sáng tỏ cơ chế tác động của bước tiền xử lý văn bản tiếng Việt và bộ lọc âm học đến độ chính xác phát âm zero-shot.
- Cung cấp cơ sở khoa học để đề xuất khuyến nghị cải tiến cấu trúc pipeline tổng hợp giọng nói tiếng Việt cho các hệ thống TTS thế hệ mới.
- Xuất bản báo cáo so sánh đa phương tiện HTML và tập dữ liệu âm thanh đối chứng.


In [ ]:
# Ô 1 · MÔI TRƯỜNG, INPUT VÀ THIẾT LẬP
import os,sys,time,subprocess,importlib,importlib.util,importlib.metadata,json,hashlib,random,shutil,base64,io,html,uuid
from pathlib import Path
for k in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):os.environ[k]='1'
def can(pkg,mod):
    if importlib.util.find_spec(mod) is not None:print('✓',pkg);return
    print('Cài',pkg,'— cần Internet',flush=True)
    r=subprocess.run([sys.executable,'-m','pip','install','-q',pkg],capture_output=True,text=True);importlib.invalidate_caches()
    if r.returncode!=0 or importlib.util.find_spec(mod) is None:raise RuntimeError('Không cài được '+pkg+': '+(r.stderr or r.stdout)[-1500:])
for p,m in [('numpy','numpy'),('pandas','pandas'),('soundfile','soundfile'),('scipy','scipy'),('matplotlib','matplotlib'),('torch','torch'),('omnivoice','omnivoice'),('huggingface_hub','huggingface_hub')]:can(p,m)
import numpy as np,pandas as pd,soundfile as sf,torch,scipy,matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import resample_poly
from math import gcd
from huggingface_hub import snapshot_download
from omnivoice import OmniVoice
NGUON=Path('/kaggle/input/notebooks/ta-vivos-thatgia-tudong/ta_vivos')
for n in ['manifest_TA.csv','tham_chieu.csv','nhat_ky_sinh.json','thong_tin_TA.json','thiet_ke_cap.csv']:
    if not (NGUON/n).is_file():raise FileNotFoundError('Không thấy '+str(NGUON/n)+'. Add Input output notebook ta-vivos-thatgia-tudong chứa thư mục ta_vivos.')
if not torch.cuda.is_available():raise RuntimeError('Bật GPU T4 trong Settings rồi chạy lại.')
RUN=time.strftime('%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6];RA=Path('/kaggle/working/buoc5_omni_'+RUN);RA.mkdir(parents=True,exist_ok=False)
print('✓ Input:',NGUON,'· GPU:',torch.cuda.get_device_name(0),'· Output:',RA,flush=True)


In [ ]:
# Ô 2 · 4 CÂU TỪ PHIẾU NGHE THẬT CỦA BẠN, KHÔNG CHỌN THEO DỰ ĐOÁN PHÂN LOẠI
# Câu 1 và 3 từng chấm sai; câu 9 và 13 từng chấm vừa thiếu vừa sai. Bốn người khác nhau.
fixed=[('VIVOSSPK03__VIVOSSPK03_T015','sai'),('VIVOSSPK19__VIVOSSPK19_167','sai'),('VIVOSSPK30__VIVOSSPK30_106','thieu_va_sai'),('VIVOSSPK45__VIVOSSPK45_041','thieu_va_sai')]
M=pd.read_csv(NGUON/'manifest_TA.csv');ref=pd.read_csv(NGUON/'tham_chieu.csv');oldlog=json.loads((NGUON/'nhat_ky_sinh.json').read_text(encoding='utf-8'));oldinfo=json.loads((NGUON/'thong_tin_TA.json').read_text(encoding='utf-8'))
if M.id_clip.duplicated().any():raise ValueError('Manifest cũ có id trùng')
cases=[]
for idx,(pair,old_rating) in enumerate(fixed,1):
    q=M[(M.pair_id==pair)&(M.tap=='test')]
    if len(q)!=4 or set(q.nguon)!={'that','vieneu','omnivoice','omdiom'}:raise ValueError('Thiếu 4 nguồn: '+pair)
    if q.speaker.nunique()!=1 or q.cau_id.nunique()!=1 or q.text.nunique()!=1:raise ValueError('Cặp không thống nhất: '+pair)
    r=q.iloc[0];rr=ref[ref.speaker==r.speaker]
    if len(rr)!=1:raise ValueError('Không tìm được tham chiếu duy nhất: '+r.speaker)
    path_ref=NGUON/'ref'/(r.speaker+'.wav')
    if not path_ref.is_file():raise FileNotFoundError('Thiếu WAV tham chiếu: '+str(path_ref))
    inf=sf.info(path_ref)
    if inf.samplerate!=16000 or inf.channels!=1 or inf.frames<=0:raise ValueError('WAV tham chiếu không hợp lệ: '+str(path_ref))
    sources={}
    for b in ['that','omnivoice','omdiom','vieneu']:
        path=NGUON/'audio'/b/r.speaker/(r.cau_id+'.flac')
        if not path.is_file():raise FileNotFoundError('Thiếu audio cũ: '+str(path))
        si=sf.info(path)
        if si.samplerate!=16000 or si.channels!=1 or si.frames<=0:raise ValueError('Audio cũ không hợp lệ: '+str(path))
        sources[b]=str(path)
    ok=[x for x in oldlog if x.get('pair_id')==pair and x.get('bo_sinh')=='omnivoice' and x.get('trang_thai')=='da_sinh_can_nghe']
    if len(ok)!=1 or 'seed' not in ok[0]:raise ValueError('Không thấy seed thành công duy nhất của OmniVoice: '+pair)
    seed1=int(ok[0]['seed']);seed2=(seed1+1234567)%(2**32-1)
    if seed1==seed2:raise ValueError('Seed trùng')
    cases.append({'so':idx,'pair_id':pair,'speaker':r.speaker,'cau_id':r.cau_id,'text':r.text,'ref_text':rr.iloc[0].ref_text,'ref_id':rr.iloc[0].ref_id,'ref_wav':str(path_ref),'danh_gia_cu':old_rating,'seed1':seed1,'seed2':seed2,'nguon_cu':sources})
print('✓ 4 câu / 4 người, seed thành công Bước 3 được nạp; số lượt mới dự kiến:',len(cases)*3*2)
for q in cases:print(f"  {q['so']}. {q['speaker']} / {q['cau_id']} — bạn từng chấm: {q['danh_gia_cu']}")
(RA/'bon_cau_chon.json').write_text(json.dumps(cases,ensure_ascii=False,indent=2),encoding='utf-8')


In [ ]:
# Ô 2b · NẾU CÓ TRAIN.JSONL CŨ, THĂM DÒ TRÙNG MÃ VIVOS — KHÔNG BẮT BUỘC
# Quét các đường dẫn gọn, không duyệt toàn bộ WAV. Không tự nhận một manifest là nguồn checkpoint.
import glob
patterns=['/kaggle/input/notebooks/*/*/du_lieu/train.jsonl','/kaggle/input/notebooks/*/*/du_lieu/dev.jsonl',
          '/kaggle/input/datasets/*/*/du_lieu/train.jsonl','/kaggle/input/datasets/*/*/du_lieu/dev.jsonl']
manifest_paths=sorted(set(p for pat in patterns for p in glob.glob(pat)))
ke=pd.read_csv(NGUON/'thiet_ke_cap.csv');overlap_summaries=[];overlap_rows=[]
for file in manifest_paths:
    ids=set();bad=0
    try:
        with open(file,'r',encoding='utf-8-sig') as f:
            for line in f:
                if not line.strip():continue
                try:
                    obj=json.loads(line);raw=obj.get('id') or obj.get('audio_path') or obj.get('audio') or ''
                    key=Path(str(raw).replace(chr(92),'/')).stem
                    if key:ids.add(key)
                    else:bad+=1
                except Exception:bad+=1
        speakers={x.split('_')[0] for x in ids if x.startswith('VIVOSSPK')}
        for tap,g in ke.groupby('tap'):
            target=set(g.cau_id.astype(str));references=set(g.ref_id.astype(str));people=set(g.speaker.astype(str))
            overlap_summaries.append({'file_manifest':file,'vai_tro_manifest':Path(file).name,'tap_phan_loai':tap,'so_id_manifest':len(ids),'loi_dong':bad,'so_cau_dich':len(target),'cau_dich_trung_id':len(target&ids),'so_nguoi':len(people),'nguoi_trung_ma':len(people&speakers),'so_ref':len(references),'ref_trung_id':len(references&ids),'chu_y':'Trùng ID không xác nhận trùng nội dung; chưa xác nhận manifest này tạo checkpoint đang dùng.'})
        for r in ke.itertuples(index=False):
            overlap_rows.append({'file_manifest':file,'tap_phan_loai':r.tap,'speaker':r.speaker,'pair_id':r.pair_id,'cau_id':r.cau_id,'cau_trung_id':r.cau_id in ids,'ref_id':r.ref_id,'ref_trung_id':r.ref_id in ids,'speaker_trung_ma':r.speaker in speakers})
    except Exception as e:overlap_summaries.append({'file_manifest':file,'loi':f'{type(e).__name__}: {str(e)[:350]}'})
if overlap_summaries:
    pd.DataFrame(overlap_summaries).to_csv(RA/'tom_tat_doi_chieu_manifest.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(overlap_rows).to_csv(RA/'doi_chieu_tung_ma.csv',index=False,encoding='utf-8-sig')
    overlap_msg=f'Đã thăm dò {len(manifest_paths)} file train/dev. Phải đối chiếu nguồn checkpoint trước khi kết luận; trùng ID chưa chứng minh trùng bản ghi.'
else:
    overlap_msg='Không thấy du_lieu/train.jsonl hoặc dev.jsonl trong Input. Chưa xác định được mức trùng từng câu/người với fine-tune TTS.'
(RA/'trang_thai_doi_chieu_manifest.txt').write_text(overlap_msg,encoding='utf-8')
print('✓',overlap_msg)


In [ ]:
# Ô 3 · NẠP OMNIVOICE GỐC VÀ GHI DẤU VẾT PHIÊN BẢN
# Không có phiên bản/snapshot cũ trong metadata Bước 3 nên không giả vờ khớp tuyệt đối.
def ver(pkg):
    try:return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:return 'không xác định'
versions={k:ver(k) for k in ['omnivoice','torch','transformers','huggingface-hub','numpy','scipy','soundfile']}
versions.update({'python':sys.version,'gpu':torch.cuda.get_device_name(0),'nguon_model':'k2-fsa/OmniVoice','tham_so_chung':{'language':'vi','num_step':32,'prompt':'ref 16 kHz và ref_text gốc','so_seed':2},'luu_y':'Không biết revision/phiên bản chính xác của lần Bước 3; seed giống không bảo đảm tái tạo bit-exact.'})
model=None;load_error='';native_sr=None
try:
    print('Đang tải hoặc dùng cache model OmniVoice gốc...',flush=True)
    snap=snapshot_download('k2-fsa/OmniVoice');versions['snapshot_path']=str(snap);versions['snapshot_revision']=Path(snap).name
    model=OmniVoice.from_pretrained(snap,device_map='cuda',dtype=torch.float16)
    native_sr=int(getattr(model,'sampling_rate',0) or 24000);versions['sample_rate_model']=native_sr
    print('✓ OmniVoice:',versions['snapshot_revision'],'· sample rate:',native_sr,flush=True)
except Exception as e:
    import traceback
    load_error=traceback.format_exc();versions['loi_nap_model']=f'{type(e).__name__}: {str(e)[:400]}'
    print('⚠️ Không nạp được OmniVoice:',versions['loi_nap_model'],'— vẫn ghi báo cáo với trạng thái thất bại.',flush=True)
(RA/'moi_truong.json').write_text(json.dumps(versions,ensure_ascii=False,indent=2),encoding='utf-8')


In [ ]:
# Ô 4 · 24 LƯỢT ĐỐI CHỨNG CÓ KIỂM SOÁT — KHÔNG LỌC NGẮN, KHÔNG CHỌN LẠI
conditions=[('goc',True,'Giữ nguyên chữ mục tiêu; xử lý đầu ra bật'),('chu_thuong',True,'Chỉ chuyển chữ mục tiêu sang chữ thường'),('khong_xu_ly',False,'Chữ mục tiêu giữ nguyên; tắt xử lý đầu ra')]
rows=[];prompt_cache={};start=time.time()
def save_log():
    temp=RA/'nhat_ky_thu.json.tmp';temp.write_text(json.dumps(rows,ensure_ascii=False,indent=2),encoding='utf-8');os.replace(temp,RA/'nhat_ky_thu.json')
def mono(x):
    # OmniVoice trả list chứa các waveform, không lấy trung bình các câu.
    if isinstance(x,(list,tuple)):
        if len(x)!=1:raise ValueError('Đầu ra batch không có đúng một waveform')
        x=x[0]
    if hasattr(x,'detach'):x=x.detach().float().cpu().numpy()
    x=np.asarray(x,dtype=np.float32)
    if x.ndim==2 and 1 in x.shape:x=x.reshape(-1)
    if x.ndim!=1 or not x.size or not np.isfinite(x).all():raise ValueError('Âm rỗng, sai kích thước hoặc NaN/Inf')
    return x

def path16(q,k,condition):return RA/'audio_16k'/q['speaker']/q['cau_id']/f'seed{k}_{condition}.flac'
def path_native(q,k,condition):return RA/'audio_native'/q['speaker']/q['cau_id']/f'seed{k}_{condition}.wav'
def write_atomic(path,data,sr,fmt,subtype):
    path.parent.mkdir(parents=True,exist_ok=True);tmp=Path(str(path)+'.tmp');sf.write(tmp,data,sr,format=fmt,subtype=subtype)
    inf=sf.info(tmp)
    if inf.frames<=0 or inf.samplerate!=sr:raise ValueError('Lỗi xác thực WAV/FLAC vừa ghi')
    os.replace(tmp,path)

for q in cases:
    prompt=None;prompt_error=''
    try:
        if model is None:raise RuntimeError('Không nạp được model: '+versions.get('loi_nap_model','không rõ'))
        with torch.inference_mode():prompt=model.create_voice_clone_prompt(ref_audio=q['ref_wav'],ref_text=q['ref_text'])
    except Exception as e:prompt_error=f'{type(e).__name__}: {str(e)[:350]}'
    for k in [1,2]:
        seed=q['seed'+str(k)]
        for condition,post,note in conditions:
            txt=q['text'].lower() if condition=='chu_thuong' else q['text']
            r={'pair_id':q['pair_id'],'speaker':q['speaker'],'cau_id':q['cau_id'],'so_cau':q['so'],'seed_luot':k,'seed':seed,'dieu_kien':condition,'postprocess_output':post,'van_ban_muc_tieu':txt,'mo_ta':note,'trang_thai':'loi','loi':'','duong_dan_16k':'','duong_dan_native':''}
            try:
                if prompt_error:raise RuntimeError('Lỗi prompt/nạp model: '+prompt_error)
                torch.manual_seed(seed);torch.cuda.manual_seed_all(seed);np.random.seed(seed)
                with torch.inference_mode():out=model.generate(text=txt,language='vi',voice_clone_prompt=prompt,num_step=32,postprocess_output=post)
                x=mono(out);out_sr=int(native_sr)
                p24=path_native(q,k,condition);p16=path16(q,k,condition)
                write_atomic(p24,x,out_sr,'WAV','FLOAT')
                if out_sr==16000:y=x
                else:
                    g=gcd(out_sr,16000);y=resample_poly(x,16000//g,out_sr//g).astype(np.float32)
                # FLAC PCM16 áp dụng đúng chuỗi chuẩn hóa/lưu như Bước 3.
                write_atomic(p16,y,16000,'FLAC','PCM_16')
                saved,sr=sf.read(p16,dtype='float32');r.update(trang_thai='da_ghi_can_nghe',duong_dan_16k=str(p16),duong_dan_native=str(p24),
                    sr_native=out_sr,giay_16k=float(len(saved)/sr),rms_16k=float(np.sqrt(np.mean(saved*saved))),
                    peak_native=float(np.max(np.abs(x))),ti_le_vuot_bien_native=float(np.mean(np.abs(x)>1.0)),
                    sha256_16k=hashlib.sha256(p16.read_bytes()).hexdigest())
            except Exception as e:r['loi']=f'{type(e).__name__}: {str(e)[:400]}'
            rows.append(r);save_log()
            print(f"  {q['so']}/4 · seed {k} · {condition}: {r['trang_thai']} {r['loi'][:100]}",flush=True)
Q=pd.DataFrame(rows);Q.to_csv(RA/'manifest_thu.csv',index=False,encoding='utf-8-sig')
print('✓ Đã ghi',len(rows),'lượt;',sum(x['trang_thai']=='da_ghi_can_nghe' for x in rows),'có audio;',sum(x['trang_thai']=='loi' for x in rows),'lỗi. Không dùng ngưỡng thời lượng để loại.')


In [ ]:
# Ô 5 · ĐỐI CHIẾU BẢN CŨ VỚI ĐỐI CHỨNG MỚI DÙNG SEED CŨ
compare=[]
for q in cases:
    x=Q[(Q.pair_id==q['pair_id'])&(Q.seed_luot==1)&(Q.dieu_kien=='goc')].iloc[0]
    orig=Path(q['nguon_cu']['omnivoice']);rec={'pair_id':q['pair_id'],'seed_cu':q['seed1'],'sha_tep_cu':hashlib.sha256(orig.read_bytes()).hexdigest(),'trang_thai_moi':x.trang_thai}
    if x.trang_thai=='da_ghi_can_nghe':
        new=Path(x.duong_dan_16k);a,sa=sf.read(orig,dtype='float32');b,sb=sf.read(new,dtype='float32')
        rec.update(sha_tep_moi=hashlib.sha256(new.read_bytes()).hexdigest(),giong_het_mau_pcm=bool(sa==sb and a.shape==b.shape and np.array_equal(a,b)),giay_cu=float(len(a)/sa),giay_moi=float(len(b)/sb))
        if sa==sb and a.shape==b.shape and np.std(a)>1e-8 and np.std(b)>1e-8:rec['tuong_quan_mau']=float(np.corrcoef(a,b)[0,1])
    compare.append(rec)
C=pd.DataFrame(compare);C.to_csv(RA/'doi_chieu_goc_va_seed_cu.csv',index=False,encoding='utf-8-sig')
print('✓ Đối chiếu',len(C),'câu. Khác audio không tự chứng minh code sai: Bước 3 không ghi phiên bản model/thư viện.')


In [ ]:
# Ô 6 · PHIẾU NGHE A/B/C, CÓ THỂ LƯU TIẾN ĐỘ VÀ XUẤT CSV
# A/B/C xáo trộn riêng theo mỗi người và mỗi lượt seed; chỉ mở mã khi hoàn tất.
style="""body{margin:0;background:#f1f6f8;color:#19353e;font:16px/1.5 system-ui,Segoe UI,sans-serif}header{background:linear-gradient(120deg,#123d4e,#277b7d);color:white;padding:34px 22px}header div,main{max-width:1100px;margin:auto}main{padding:20px 20px 70px}h1{margin:0}h2{color:#176575}.notice{background:#fff4dd;border-left:4px solid #df9e24;padding:15px}.card{background:white;border:1px solid #dbe7eb;border-radius:13px;padding:18px;margin:16px 0}.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:12px}.box{border:1px solid #d9e6ea;background:#f9fcfd;padding:12px;border-radius:9px}audio,select,textarea{width:100%;box-sizing:border-box;margin:5px 0;font:inherit}select,textarea{padding:7px;border:1px solid #b9cfd5;border-radius:6px;background:white}textarea{height:58px}label{display:block}button{background:#12697b;color:white;border:0;border-radius:6px;padding:10px 14px;cursor:pointer;margin:5px}button.secondary{background:#4a6671}button:disabled{opacity:.5}.status{position:sticky;top:0;z-index:2;background:#dff1ec;border:1px solid #acdcd2;padding:10px}.bad{color:#9b332e}.small{font-size:14px;color:#4c6570}.reveal{font-size:14px;color:#853e2b;overflow-wrap:anywhere}"""
esc=html.escape
h=['<!doctype html><html lang="vi"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Nghe chẩn đoán OmniVoice</title><style>',style,'</style></head><body><header><div><h1>Nghe chẩn đoán OmniVoice</h1><p>4 câu đã nghe · 3 điều kiện × 2 seed · nghe đối chứng</p></div></header><main>']
h.append('<div class="notice"><b>Không phải benchmark:</b> Bốn câu đã được chọn vì bạn nghe thấy lỗi. Cùng câu và seed, A/B/C xáo trộn riêng. Nghe tham chiếu và transcript trước; nghe bản thật/bản sinh cũ nếu cần; sau đó chấm từng bản mới. A ở nhóm này không nhất thiết trùng điều kiện với A ở nhóm khác. Kết quả chỉ là nhận xét chủ quan và chưa chứng minh nguyên nhân.</div>')
h.append('<div class="card"><div id="status" class="status"></div><button id="save">Lưu tiến độ JSON</button><button id="importBtn" class="secondary">Mở tiến độ JSON</button><input type="file" id="importFile" accept=".json,application/json" style="display:none"><button id="export" disabled>Hoàn tất & tải kết quả CSV</button><button id="show" class="secondary" disabled>Hiện mã điều kiện</button><p class="small">Cần chấm 4 tham chiếu và mọi clip mới sinh thành công. Các lượt lỗi sinh tự được ghi là lỗi, không bị loại khỏi CSV. Nếu không chắc, chọn “Không rõ”. CSV làm lộ mã điều kiện.</p><div id="reveal" class="reveal"></div></div>')
opts=[('','— Chọn —'),('du_dung','Đủ và đúng chữ'),('sai','Sai chữ'),('thieu','Thiếu chữ'),('thieu_va_sai','Vừa thiếu vừa sai'),('khong_ro','Không rõ')]
refopts=[('','— Chọn —'),('dung','Khớp transcript'),('sai_thieu','Có vẻ sai / thiếu'),('khong_ro','Không rõ')]
def opt(v):return ''.join('<option value="'+esc(a,quote=True)+'">'+esc(b)+'</option>' for a,b in v)
def audio(path,mime):return 'data:'+mime+';base64,'+base64.b64encode(Path(path).read_bytes()).decode('ascii')
shuffle=random.Random(20261105);html_meta=[]
for q in cases:
    h.append('<section class="card"><h2>Câu '+str(q['so'])+' · '+esc(q['speaker'])+' · '+esc(q['cau_id'])+'</h2><p><b>Câu mục tiêu:</b> '+esc(q['text'])+'</p>')
    h.append('<div class="box"><b>Tham chiếu giọng — '+esc(q['ref_id'])+'</b><p>Transcript tham chiếu: '+esc(q['ref_text'])+'</p><audio controls preload="none" src="'+audio(q['ref_wav'],'audio/wav')+'"></audio><label>Nghe tham chiếu có khớp transcript không?<select data-kind="ref" data-key="'+esc(q['pair_id'],quote=True)+'">'+opt(refopts)+'</select></label></div>')
    h.append('<h3>Các bản cũ — cùng câu để đối chiếu</h3><div class="grid">')
    for b,lab in [('that','Bản thật'),('omnivoice','OmniVoice cũ'),('omdiom','OmDiOm cũ'),('vieneu','VieNeu cũ')]:
        h.append('<div class="box"><b>'+lab+'</b><audio controls preload="none" src="'+audio(q['nguon_cu'][b],'audio/flac')+'"></audio></div>')
    h.append('</div>')
    for k in [1,2]:
        z=Q[(Q.pair_id==q['pair_id'])&(Q.seed_luot==k)];assert len(z)==3
        order=['goc','chu_thuong','khong_xu_ly'];shuffle.shuffle(order)
        h.append('<h3>Lượt seed '+str(k)+' — ba điều kiện A/B/C</h3><div class="grid">')
        for letter,condition in zip(['A','B','C'],order):
            r=z[z.dieu_kien==condition].iloc[0];key=q['pair_id']+'|'+str(k)+'|'+letter
            success=r.trang_thai=='da_ghi_can_nghe'
            item={'key':key,'pair_id':q['pair_id'],'speaker':q['speaker'],'cau_id':q['cau_id'],'so_cau':q['so'],'text':q['text'],'seed_luot':k,'seed':int(r.seed),'nhan_an':letter,'dieu_kien':condition,'trang_thai':str(r.trang_thai),'loi':str(r.loi or '')}
            html_meta.append(item)
            h.append('<div class="box"><b>Bản '+letter+'</b>')
            if success:
                h.append('<audio controls preload="none" src="'+audio(r.duong_dan_16k,'audio/flac')+'"></audio><p class="small">'+f"{r.giay_16k:.2f} giây · nghe để đánh giá nội dung"+'</p><label>Nội dung so với câu mục tiêu<select data-kind="clip" data-key="'+esc(key,quote=True)+'">'+opt(opts)+'</select></label><label>Ghi chú từ thiếu/sai (tùy chọn)<textarea data-kind="note" data-key="'+esc(key,quote=True)+'" placeholder="Ví dụ: thiếu cụm ..."></textarea></label>')
            else:h.append('<p class="bad">Lượt này không sinh được audio; ghi là lỗi trong CSV. '+esc(str(r.loi))+'</p>')
            h.append('</div>')
        h.append('</div>')
    h.append('</section>')
eval_id=hashlib.sha256(json.dumps(html_meta,sort_keys=True,ensure_ascii=False).encode('utf8')).hexdigest()[:16]
jsmeta=json.dumps(html_meta,ensure_ascii=False).replace('<','\\u003c');jsref=json.dumps([{'pair_id':q['pair_id'],'ref_id':q['ref_id'],'ref_text':q['ref_text']} for q in cases],ensure_ascii=False).replace('<','\\u003c')
script=r"""
<script>
const ITEMS=__ITEMS__, REFS=__REFS__, EVAL_ID=__ID__, STORAGE='BUOC5_'+EVAL_ID;
let state={eval_id:EVAL_ID,clips:{},refs:{},revealed:false};
try{const t=JSON.parse(localStorage.getItem(STORAGE)||'null');if(t&&t.eval_id===EVAL_ID)state=t;}catch(e){}
const valid=ITEMS.filter(x=>x.trang_thai==='da_ghi_can_nghe');
function persist(){try{localStorage.setItem(STORAGE,JSON.stringify(state))}catch(e){}}
function render(){
 document.querySelectorAll('[data-kind]').forEach(e=>{const k=e.dataset.key,kind=e.dataset.kind;let val=kind==='ref'?state.refs[k]:(kind==='clip'?(state.clips[k]||{}).noi_dung:(state.clips[k]||{}).ghi_chu);e.value=val||''});
 const a=valid.filter(x=>(state.clips[x.key]||{}).noi_dung).length,b=REFS.filter(x=>state.refs[x.pair_id]).length;
 document.getElementById('status').textContent='Đã chấm '+a+'/'+valid.length+' clip sinh thành công và '+b+'/'+REFS.length+' clip tham chiếu · Mã phiếu '+EVAL_ID;
 const ready=(a===valid.length && b===REFS.length);document.getElementById('export').disabled=!ready;document.getElementById('show').disabled=!ready;if(state.revealed)reveal();
}
document.querySelectorAll('[data-kind]').forEach(e=>e.addEventListener('change',()=>{let k=e.dataset.key,t=e.dataset.kind;if(t==='ref')state.refs[k]=e.value;else{state.clips[k]=state.clips[k]||{};state.clips[k][t==='clip'?'noi_dung':'ghi_chu']=e.value}persist();render()}));
function download(text,name,mime){const u=URL.createObjectURL(new Blob([text],{type:mime})),a=document.createElement('a');a.href=u;a.download=name;document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(u),10000)}
document.getElementById('save').onclick=()=>download(JSON.stringify(state,null,2),'tien_do_buoc5_'+EVAL_ID+'.json','application/json;charset=utf-8');
document.getElementById('importBtn').onclick=()=>document.getElementById('importFile').click();
document.getElementById('importFile').onchange=e=>{let f=e.target.files[0];if(!f)return;let r=new FileReader();r.onload=()=>{try{let a=JSON.parse(r.result);if(a.eval_id!==EVAL_ID||!a.clips||!a.refs)throw Error('Mã phiếu không khớp');state=a;persist();render();alert('Đã mở tiến độ')}catch(err){alert('Không mở được: '+err.message)}};r.readAsText(f,'UTF-8');e.target.value='';};
function csv(s){let t=String(s??'');if(/^[\s]*[=+@-]/.test(t))t="'"+t;return '"'+t.replace(/"/g,'""')+'"';}
document.getElementById('export').onclick=()=>{
 let rows=[['eval_id','so_cau','pair_id','speaker','cau_id','text','ref_id','ref_text','tham_chieu_khop','seed_luot','seed','nhan_an','dieu_kien','trang_thai_sinh','loi_sinh','danh_gia_noi_dung','ghi_chu']];
 for(let q of ITEMS){let a=state.clips[q.key]||{},rf=REFS.find(x=>x.pair_id===q.pair_id);rows.push([EVAL_ID,q.so_cau,q.pair_id,q.speaker,q.cau_id,q.text,rf.ref_id,rf.ref_text,state.refs[q.pair_id]||'',q.seed_luot,q.seed,q.nhan_an,q.dieu_kien,q.trang_thai,q.loi,q.trang_thai==='da_ghi_can_nghe'?(a.noi_dung||''):'loi_sinh',a.ghi_chu||'']);}
 download('\uFEFF'+rows.map(r=>r.map(csv).join(',')).join('\r\n'),'ket_qua_nghe_buoc5_'+EVAL_ID+'.csv','text/csv;charset=utf-8');state.revealed=true;persist();reveal();
};
function reveal(){document.getElementById('reveal').textContent=ITEMS.map(q=>'Câu '+q.so_cau+' / lượt '+q.seed_luot+' / '+q.nhan_an+' = '+q.dieu_kien).join(' | ')}
document.getElementById('show').onclick=()=>{if(confirm('Hiện mã sẽ mất ẩn nhãn. Đã hoàn thành phiếu?')){state.revealed=true;persist();reveal()}};
render();
</script>"""
script=script.replace('__ITEMS__',jsmeta).replace('__REFS__',jsref).replace('__ID__',json.dumps(eval_id))
h.append(script+'</main></body></html>');listen=RA/'NGHE_CHAN_DOAN_OMNIVOICE.html';listen.write_text(''.join(h),encoding='utf-8')
print(f'✓ HTML nghe: {listen} · {len(html_meta)} lượt sinh, {sum(x["trang_thai"]=="da_ghi_can_nghe" for x in html_meta)} có audio · mã {eval_id}')


In [ ]:
# Ô 7 · BÁO CÁO KỸ THUẬT, HÌNH VÀ BẢNG — KHÔNG BỊA ĐÁNH GIÁ NGHE
plt.rcParams.update({'figure.facecolor':'white','axes.spines.top':False,'axes.spines.right':False,'font.size':10.5,'axes.titleweight':'bold','figure.autolayout':True});figs=[];(RA/'hinh').mkdir(exist_ok=True)
def plot(title,note,fig,name):
    b=io.BytesIO();fig.savefig(b,format='png',dpi=145,bbox_inches='tight',facecolor='white');plt.close(fig);data=b.getvalue();(RA/'hinh'/(name+'.png')).write_bytes(data);figs.append((title,note,base64.b64encode(data).decode('ascii')))
fig,axes=plt.subplots(2,2,figsize=(10,6.6));col={'goc':'#237e97','chu_thuong':'#e29359','khong_xu_ly':'#42a383'}
for ax,q in zip(axes.flat,cases):
    z=Q[Q.pair_id==q['pair_id']];pos=np.arange(2);w=.22
    for ix,(cond,post,note) in enumerate(conditions):
        val=[]
        for k in [1,2]:
            r=z[(z.seed_luot==k)&(z.dieu_kien==cond)].iloc[0];val.append(float(r.giay_16k) if r.trang_thai=='da_ghi_can_nghe' else np.nan)
        ax.bar(pos+(ix-1)*w,val,width=w,label=cond,color=col[cond])
    old=sf.info(q['nguon_cu']['omnivoice']);truth=sf.info(q['nguon_cu']['that']);ax.axhline(old.frames/old.samplerate,ls='--',color='#c36c55',lw=1.2,label='Omni cũ');ax.axhline(truth.frames/truth.samplerate,ls=':',color='#384e5b',lw=1.3,label='thật')
    ax.set_title('Câu '+str(q['so'])+' · '+q['speaker']);ax.set_xticks(pos,['seed cũ','seed thứ hai']);ax.set_ylabel('giây');ax.grid(axis='y',alpha=.15)
axes.flat[0].legend(frameon=False,fontsize=7,loc='upper right')
plot('Thời lượng từng lượt','Thời lượng khác nhau không đủ để kết luận câu có bị thiếu hoặc sai chữ. Seed cũ không bảo đảm âm thanh tái tạo bit-exact.',fig,'01_thoi_luong')
fig,ax=plt.subplots(figsize=(7,3.7));counts=Q.groupby(['dieu_kien','trang_thai']).size().unstack(fill_value=0).reindex([x[0] for x in conditions],fill_value=0)
for status,color in [('da_ghi_can_nghe','#2b9d89'),('loi','#c95f56')]:
 if status not in counts:counts[status]=0
left=np.zeros(len(counts))
for status,lab,color in [('da_ghi_can_nghe','Có tệp cần nghe','#2b9d89'),('loi','Lỗi sinh','#c95f56')]:
 vals=counts[status].to_numpy();ax.barh(counts.index,vals,left=left,label=lab,color=color);left+=vals
ax.set_xlim(0,8.5);ax.set_xlabel('Số lượt / 8');ax.set_title('Lượt có tệp và lượt lỗi');ax.legend(frameon=False)
plot('Lượt sinh','Không tự động loại clip ngắn/nhỏ hay chọn bản hay nhất. Có tệp không có nghĩa là đọc đúng.',fig,'02_trang_thai')
style="""body{margin:0;background:#f2f6f8;color:#18323c;font:16px/1.55 system-ui,Segoe UI,sans-serif}header{background:linear-gradient(125deg,#123d4e,#207d80);color:white;padding:36px 20px}header div,main{max-width:1100px;margin:auto}main{padding:20px 20px 60px}h1{margin:0}h2{margin-top:36px;color:#176575}figure,.card{background:white;border:1px solid #dce7eb;border-radius:12px;padding:18px;overflow:auto;margin:16px 0}figure img{max-width:100%;display:block;margin:auto}figcaption{border-left:3px solid #3599a3;padding-left:12px;color:#526a72;font-size:14px}.warn{background:#fff5df;border-left:4px solid #e2a935;padding:16px}.table{border-collapse:collapse;white-space:nowrap;font-size:13px}.table td,.table th{padding:7px;border-bottom:1px solid #dfe9eb}.table th{background:#e7f3f4}"""
def table(df):return df.to_html(index=False,classes='table',border=0,escape=True,float_format=lambda x:f'{x:.4f}')
H=['<!doctype html><html lang="vi"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Chẩn đoán OmniVoice</title><style>',style,'</style></head><body><header><div><h1>Chẩn đoán OmniVoice đọc sai / thiếu chữ</h1><p>4 câu có chủ đích · 3 điều kiện · 2 seed · 24 lượt</p></div></header><main>']
H.append('<div class="warn"><b>Giới hạn:</b> Chọn bốn câu từ test đã thấy lỗi theo nhận xét của một người, nên đây chỉ là thăm dò nguyên nhân. Không dùng kết quả này để chọn mô hình hoặc thay thế điểm test T-A. Không đảm bảo tái tạo bit-exact bản cũ vì không có revision/phiên bản cũ.</div>')
H.append('<h2>Thiết kế</h2><div class="card"><p>Gốc: nguyên văn, bật postprocess; chữ thường: chỉ chuyển văn bản mục tiêu sang chữ thường; không xử lý: nguyên văn, tắt postprocess. Cùng câu/lượt dùng cùng seed, tham chiếu và num_step=32. Không lọc clip ngắn/nhỏ, không thử lại đến khi được.</p><p>Một seed dựa trên nhật ký Bước 3, seed thứ hai cố định khác. Kết quả nghe phải được ghi riêng trong CSV từ NGHE_CHAN_DOAN_OMNIVOICE.html; chưa có số liệu đánh giá nghe mới.</p></div>')
H.append('<h2>Tình trạng</h2><div class="card"><p>'+str(len(Q))+' lượt: '+str(sum(Q.trang_thai=='da_ghi_can_nghe'))+' có tệp cần nghe; '+str(sum(Q.trang_thai=='loi'))+' lỗi sinh. Có tệp không đồng nghĩa đọc đúng. WAV native là float; FLAC 16 kHz được tạo để nghe và so với Bước 3.</p></div>')
H.append('<h2>Đối chiếu seed cũ</h2><div class="card">'+table(C)+'</div>')
H.append('<h2>Biểu đồ</h2>')
for title,note,img in figs:H.append('<figure><h3>'+html.escape(title)+'</h3><img src="data:image/png;base64,'+img+'"><figcaption>'+html.escape(note)+'</figcaption></figure>')
H.append('<h2>Đối chiếu dữ liệu fine-tune</h2><div class="card"><p>'+html.escape(overlap_msg)+'</p></div>')
H.append('<h2>Phiên bản và xuất xứ</h2><div class="card"><pre>'+html.escape(json.dumps(versions,ensure_ascii=False,indent=2))+'</pre><p>Các bản OmDiOm được fine-tune trên VIVOS theo xác nhận người dùng; bốn câu chọn từ VIVOS test của bộ phân loại, không bảo đảm chưa gặp trong huấn luyện TTS. Nguồn bộ sinh OmniVoice gốc: k2-fsa/OmniVoice.</p></div></main></body></html>')
report=RA/'BAO_CAO_BUOC5.html';report.write_text(''.join(H),encoding='utf-8');print('✓ Báo cáo kỹ thuật:',report,'·',len(figs),'hình')


In [ ]:
# Ô 8 · HƯỚNG DẪN VÀ ZIP
huongdan="""# Bước 5 · Cách sử dụng

1. Tải NGHE_CHAN_DOAN_OMNIVOICE.html về máy, mở bằng Chrome/Edge. Mỗi câu nghe tham chiếu và transcript, các bản cũ, rồi lần lượt A/B/C của hai seed. A/B/C đổi thứ tự mỗi nhóm.
2. Chấm tham chiếu có khớp transcript, chấm nội dung từng clip mới. Nếu không chắc chọn Không rõ. Lượt sinh lỗi được giữ trong CSV, không phải tự chọn bản khác.
3. Khi cần nghỉ chọn Lưu tiến độ JSON. Mở lại bằng Mở tiến độ JSON. Sau khi chấm xong, nhấn Hoàn tất & tải kết quả CSV, rồi gửi CSV cho người phân tích; không gửi HTML thay CSV.
4. BAO_CAO_BUOC5.html cho thấy số lượt sinh và thời lượng, không tự khẳng định nội dung đúng. `manifest_thu.csv`, `nhat_ky_thu.json`, `moi_truong.json`, `doi_chieu_goc_va_seed_cu.csv` là bằng chứng kỹ thuật. Nếu không nạp được model, gửi các tệp log và báo cáo.
5. Nếu có Input output huấn luyện cũ với `du_lieu/train.jsonl`, `du_lieu/dev.jsonl`, notebook đối chiếu mã VIVOS; chưa chứng minh các manifest này đã dùng cho checkpoint. Nếu không có, ghi chưa xác định mức trùng.
6. `audio_native/` là WAV float tại sample rate model, `audio_16k/` là FLAC PCM16 được chuẩn hóa theo cách Bước 3 để nghe. Không lấy bản ngắn/lỗi ra khỏi phân tích và không chọn chỉ seed tốt.

Đây là nghiên cứu thăm dò trên 4 câu đã biết có lỗi, không phải test độc lập. Kết quả không thay đổi T-A gốc và không đủ khẳng định nguyên nhân lỗi. Người nghe có thể nhận ra giọng/điều kiện nên không gọi đây là thử mù nghiêm ngặt.
"""
(RA/'HUONG_DAN_BUOC5.md').write_text(huongdan,encoding='utf8')
(RA/'thong_tin_buoc5.json').write_text(json.dumps({'eval_id':eval_id,'so_cau':len(cases),'luot_du_kien':24,'luot_sinh_co_file':int(sum(Q.trang_thai=='da_ghi_can_nghe')),'luot_loi':int(sum(Q.trang_thai=='loi')),'nguon_buoc3':str(NGUON),'thiet_ke':'4 cau da biet loi; 3 dieu kien; 2 seed; tham do, khong chon theo test','omdiom_da_finetune_vivos':'nguoi dung xac nhan'},ensure_ascii=False,indent=2),encoding='utf8')
archive=shutil.make_archive('/kaggle/working/KET_QUA_BUOC5_OMNI_'+RUN,'zip',RA)
print('\n📄 Phiếu nghe:',listen,'\n📊 Báo cáo:',report,'\n📦 ZIP:',archive, f'({Path(archive).stat().st_size/1024**2:.1f} MiB)')
print('Chấm nghe rồi gửi ket_qua_nghe_buoc5_<ma>.csv; nếu không sinh được, gửi nhat_ky_thu.json và moi_truong.json.')
